<a href="https://colab.research.google.com/github/Skamal03/Security-Intent-Operation-Oriented-Chatbot-for-SafeScan/blob/main/SafeScan_Phi3_Finetune_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔐 SafeScan × Phi-3 Mini 3.8B — Fine-Tuning Notebook

**Model:** `microsoft/Phi-3-mini-4k-instruct`  
**Dataset:** `SafeScan_Training.jsonl` (1250 rows, intent-routing JSON completions)  
**Method:** QLoRA (4-bit quantization + LoRA adapters) via `trl` + `peft`  
**Target:** Push fine-tuned model to Hugging Face Hub

---
### ⚙️ Before running:
1. `Runtime → Change runtime type → T4 GPU` (free tier works, A100 is faster)
2. Have your **Hugging Face token** (write access) ready
3. Upload `SafeScan_Training.jsonl` when prompted in Cell 4


## Cell 1 — Install Dependencies

In [ ]:
# Install all required packages
# This takes ~2-3 minutes on a fresh Colab runtime
!pip install -q \
    transformers==4.44.2 \
    trl==0.10.1 \
    peft==0.12.0 \
    accelerate==0.34.2 \
    bitsandbytes==0.43.3 \
    datasets==2.21.0 \
    huggingface_hub==0.24.7 \
    scipy \
    einops

print('✅ All packages installed.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 122.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.1/280.1 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.5/417.5 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 96.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 16.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behavi

## Cell 2 — Verify GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('❌ No GPU detected. Go to Runtime → Change runtime type → GPU (T4).')

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f'✅ GPU: {gpu_name}')
print(f'✅ VRAM: {vram_gb:.1f} GB')
print(f'✅ CUDA: {torch.version.cuda}')
print(f'✅ PyTorch: {torch.__version__}')

✅ GPU: Tesla T4
✅ VRAM: 15.6 GB
✅ CUDA: 12.8
✅ PyTorch: 2.11.0+cu128


## Cell 3 — Hugging Face Login

In [ ]:
from huggingface_hub import login

# You will be prompted to paste your HF token (write access required)
# Get it from: https://huggingface.co/settings/tokens
login()

# ── SET YOUR HF USERNAME HERE ──────────────────────────────────────────────
HF_USERNAME = 'MuhammadSanan99989'            # e.g. 'gypsy-dev'
MODEL_REPO_NAME = 'safescan-phi3-mini-intent'
# ──────────────────────────────────────────────────────────────────────────

FULL_REPO_ID = f'{HF_USERNAME}/{MODEL_REPO_NAME}'
print(f'✅ Will push to: https://huggingface.co/{FULL_REPO_ID}')

✅ Will push to: https://huggingface.co/MuhammadSanan99989/safescan-phi3-mini-intent


## Cell 4 — Upload & Load Dataset

In [ ]:
from google.colab import files
import json
from datasets import Dataset

print('📂 Upload your SafeScan_Training.jsonl file:')
uploaded = files.upload()

jsonl_filename = list(uploaded.keys())[0]
print(f'\n✅ Uploaded: {jsonl_filename}')

raw_data = []
with open(jsonl_filename, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            raw_data.append(json.loads(line))

print(f'✅ Loaded {len(raw_data)} rows')
print(f'\n── Sample row ──────────────────────────────')
print(f'PROMPT   : {raw_data[0]["prompt"]}')
print(f'COMPLETION: {raw_data[0]["completion"]}')

📂 Upload your SafeScan_Training.jsonl file:


Saving SafeScan_Training.jsonl to SafeScan_Training.jsonl

✅ Uploaded: SafeScan_Training.jsonl
✅ Loaded 1250 rows

── Sample row ──────────────────────────────
PROMPT   : <|user|>
Is my shared WiFi secure from hackers?<|end|>
<|assistant|>
COMPLETION: {"intent": "wifi_check", "module": "WifiSecurityScan", "action": "navigate_to_wifi_security_scan", "parameters": {"network": null}}


## Cell 5 — Format Dataset for SFTTrainer

In [ ]:
from datasets import Dataset

def format_for_sft(example):
    """
    SFTTrainer expects a single 'text' field with the full prompt+completion.
    Final format:
        <|user|>\n{user message}<|end|>\n<|assistant|>{json completion}<|end|>
    """
    prompt = example['prompt'].strip()
    completion = example['completion'].strip()
    return {'text': f'{prompt}{completion}<|end|>'}

raw_dataset = Dataset.from_list(raw_data)
formatted_dataset = raw_dataset.map(format_for_sft, remove_columns=['prompt', 'completion'])

# 95 / 5 train-eval split
split = formatted_dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = split['train']
eval_dataset  = split['test']

print(f'✅ Train samples : {len(train_dataset)}')
print(f'✅ Eval  samples : {len(eval_dataset)}')
print(f'\n── Formatted sample ─────────────────────────')
print(train_dataset[0]['text'])

Map:   0%|          | 0/1250 [00:00<?, ? examples/s]

✅ Train samples : 1187
✅ Eval  samples : 63

── Formatted sample ─────────────────────────
<|user|>
Check for man-in-the-middle attacks on this caf?? WiFi.<|end|>
<|assistant|>{"intent": "wifi_check", "module": "WifiSecurityScan", "action": "navigate_to_wifi_security_scan", "parameters": {"network": null}}<|end|>


## Cell 6 — Load Tokenizer

In [ ]:
from transformers import AutoTokenizer

BASE_MODEL = 'microsoft/Phi-3-mini-4k-instruct'

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    padding_side='right',
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f'✅ Tokenizer loaded: {BASE_MODEL}')
print(f'   Vocab size : {tokenizer.vocab_size}')
print(f'   EOS token  : {repr(tokenizer.eos_token)}')
print(f'   PAD token  : {repr(tokenizer.pad_token)}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

✅ Tokenizer loaded: microsoft/Phi-3-mini-4k-instruct
   Vocab size : 32000
   EOS token  : '<|endoftext|>'
   PAD token  : '<|endoftext|>'


#Cell 7 Fix


In [ ]:
# Fix bitsandbytes CUDA compatibility issue
!pip install -q bitsandbytes --upgrade
!pip install -q triton

import importlib
import bitsandbytes
importlib.reload(bitsandbytes)

print("✅ bitsandbytes fixed")

## Cell 7 — Load Base Model in 4-bit (QLoRA)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Downloads ~2.3 GB on first run
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.float16,
    attn_implementation='eager',
)

model.config.use_cache = False
model.config.pretraining_tp = 1

total_params = sum(p.numel() for p in model.parameters()) / 1e9
print(f'✅ Model loaded: {BASE_MODEL}')
print(f'   Total params: {total_params:.2f}B')

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

configuration_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3-mini-4k-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3-mini-4k-instruct:
- modeling_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

RuntimeError: Failed to import transformers.integrations.bitsandbytes because of the following error (look up to see its traceback):
No module named 'triton.ops'

## Cell 8 — LoRA Configuration

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)

trainable, total = model.get_nb_trainable_parameters()
print(f'✅ LoRA adapters attached')
print(f'   Trainable params : {trainable:,}  ({100 * trainable / total:.2f}% of total)')
print(f'   Frozen params    : {total - trainable:,}')

## Cell 9 — Training Arguments

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./safescan-phi3-checkpoints',

    # Epochs & batching
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,      # Effective batch = 16

    # Optimizer
    optim='paged_adamw_8bit',
    learning_rate=2e-4,
    weight_decay=0.001,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,

    # Precision — use fp16 for T4, set bf16=True and fp16=False for A100
    fp16=True,
    bf16=False,

    # Evaluation & saving
    evaluation_strategy='steps',
    eval_steps=50,
    save_strategy='steps',
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',

    # Logging
    logging_steps=10,
    report_to='none',

    # Misc
    max_grad_norm=0.3,
    group_by_length=True,
    seed=42,
)

print('✅ Training arguments configured')
print(f'   Epochs          : {training_args.num_train_epochs}')
print(f'   Batch size      : {training_args.per_device_train_batch_size}')
print(f'   Grad accum      : {training_args.gradient_accumulation_steps}')
print(f'   Effective batch : {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}')
print(f'   Learning rate   : {training_args.learning_rate}')

## Cell 10 — Initialize SFTTrainer & Train

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field='text',
    max_seq_length=512,
    peft_config=lora_config,
    args=training_args,
    packing=False,
)

print('🚀 Starting training...')

train_result = trainer.train()

print('\n✅ Training complete!')
print(f'   Final train loss : {train_result.training_loss:.4f}')
print(f'   Runtime          : {train_result.metrics["train_runtime"]:.0f}s')

## Cell 11 — Save Best Model Locally

In [ ]:
import os

LOCAL_SAVE_PATH = './safescan-phi3-lora'

trainer.save_model(LOCAL_SAVE_PATH)
tokenizer.save_pretrained(LOCAL_SAVE_PATH)

print(f'✅ LoRA adapter saved at: {LOCAL_SAVE_PATH}')
print(f'   Files: {os.listdir(LOCAL_SAVE_PATH)}')

## Cell 12 — Merge LoRA Weights into Base Model

Merging the adapters produces a standalone model that can be used for inference
without PEFT installed — which is what you want for production.

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

MERGED_PATH = './safescan-phi3-merged'

print('🔀 Loading base model in fp16 for merging (no quantization)...')
base_model_merge = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)

print('🔀 Attaching LoRA adapter...')
peft_model = PeftModel.from_pretrained(base_model_merge, LOCAL_SAVE_PATH)

print('🔀 Merging weights...')
merged_model = peft_model.merge_and_unload()

print('💾 Saving merged model...')
merged_model.save_pretrained(MERGED_PATH, safe_serialization=True)
tokenizer.save_pretrained(MERGED_PATH)

print(f'✅ Merged model saved at: {MERGED_PATH}')

## Cell 13 — Inference Sanity Check (Pre-Push)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

print('🧪 Loading merged model for inference test...')
test_model = AutoModelForCausalLM.from_pretrained(
    MERGED_PATH,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)
test_tokenizer = AutoTokenizer.from_pretrained(MERGED_PATH, trust_remote_code=True)

pipe = pipeline(
    'text-generation',
    model=test_model,
    tokenizer=test_tokenizer,
    torch_dtype=torch.float16,
    device_map='auto',
)

test_prompts = [
    '<|user|>\nIs my WiFi connection secure?<|end|>\n<|assistant|>',
    '<|user|>\nCheck SSL certificate for google.com<|end|>\n<|assistant|>',
    '<|user|>\nScan my installed apps for malware<|end|>\n<|assistant|>',
]

print('\n── Inference Results ─────────────────────────────────────────────────')
for prompt in test_prompts:
    result = pipe(
        prompt,
        max_new_tokens=128,
        do_sample=False,
        temperature=1.0,
        repetition_penalty=1.1,
        pad_token_id=test_tokenizer.pad_token_id,
        eos_token_id=test_tokenizer.convert_tokens_to_ids('<|end|>'),
    )
    generated = result[0]['generated_text'][len(prompt):].strip()
    user_msg  = prompt.split('\n')[1]
    print(f'\n  INPUT : {user_msg}')
    print(f'  OUTPUT: {generated}')

del test_model, pipe
torch.cuda.empty_cache()
print('\n✅ Sanity check done — VRAM cleared')

## Cell 14 — Push to Hugging Face Hub

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f'📤 Pushing to: https://huggingface.co/{FULL_REPO_ID}')
print('   This may take 5-10 minutes...\n')

push_model = AutoModelForCausalLM.from_pretrained(
    MERGED_PATH,
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
push_tokenizer = AutoTokenizer.from_pretrained(MERGED_PATH, trust_remote_code=True)

push_model.push_to_hub(
    FULL_REPO_ID,
    safe_serialization=True,
    private=False,           # Set True for a private repo
)
push_tokenizer.push_to_hub(FULL_REPO_ID)

print(f'\n✅ Model pushed!')
print(f'   🔗 https://huggingface.co/{FULL_REPO_ID}')

## Cell 15 — Push Model Card

In [ ]:
from huggingface_hub import ModelCard

card_content = f"""---
base_model: microsoft/Phi-3-mini-4k-instruct
language:
- en
license: mit
tags:
- phi-3
- fine-tuned
- intent-classification
- mobile-security
- flutter
- qlora
- peft
pipeline_tag: text-generation
---

# SafeScan Phi-3 Mini — Intent Routing Model

Fine-tuned version of [microsoft/Phi-3-mini-4k-instruct](https://huggingface.co/microsoft/Phi-3-mini-4k-instruct)
for the **SafeScan** mobile security utility app (Flutter).

## What it does

Given a natural language security query, returns a structured JSON object
routing the request to the correct SafeScan module:

```json
{{"intent": "wifi_check", "module": "WifiSecurityScan", "action": "navigate_to_wifi_security_scan", "parameters": {{"network": null}}}}
```

## Training Details

| Property | Value |
|---|---|
| Base model | microsoft/Phi-3-mini-4k-instruct |
| Method | QLoRA (4-bit NF4) |
| LoRA rank | 16 |
| LoRA alpha | 32 |
| Dataset size | 1,250 samples |
| Epochs | 3 |
| Learning rate | 2e-4 |
| Optimizer | paged_adamw_8bit |
| Max seq length | 512 |

## Usage

```python
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

model = AutoModelForCausalLM.from_pretrained(
    "{FULL_REPO_ID}",
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained("{FULL_REPO_ID}", trust_remote_code=True)

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

prompt = "<|user|>\\nCheck if my WiFi is secure<|end|>\\n<|assistant|>"
result = pipe(prompt, max_new_tokens=128, do_sample=False)
print(result[0]["generated_text"][len(prompt):])
# Output: {{"intent": "wifi_check", "module": "WifiSecurityScan", ...}}
```
"""

card = ModelCard(card_content)
card.push_to_hub(FULL_REPO_ID)

print('✅ Model card pushed!')
print(f'   🔗 https://huggingface.co/{FULL_REPO_ID}')

## Cell 16 — Training Loss Plot

In [ ]:
import matplotlib.pyplot as plt

log_history = trainer.state.log_history

train_steps  = [x['step'] for x in log_history if 'loss' in x and 'eval_loss' not in x]
train_losses = [x['loss'] for x in log_history if 'loss' in x and 'eval_loss' not in x]
eval_steps   = [x['step'] for x in log_history if 'eval_loss' in x]
eval_losses  = [x['eval_loss'] for x in log_history if 'eval_loss' in x]

plt.figure(figsize=(10, 5))
plt.plot(train_steps, train_losses, label='Train Loss', color='#0072B2', linewidth=2)
plt.plot(eval_steps, eval_losses, label='Eval Loss', color='#D55E00', linewidth=2, marker='o')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('SafeScan Phi-3 Fine-Tuning — Loss Curve')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('training_loss.png', dpi=150)
plt.show()
print('✅ Loss plot saved as training_loss.png')

---
## ✅ Done!

Your fine-tuned model is live on Hugging Face:

```
https://huggingface.co/{HF_USERNAME}/safescan-phi3-mini-intent
```

### What happened:
- Base Phi-3 Mini 3.8B was loaded in 4-bit (QLoRA)
- Only ~0.8% of weights were trained (LoRA adapters)
- Adapters were merged into the full model before pushing
- The pushed model needs no PEFT dependency at inference time

### Serving from Flutter:
Wrap the model in a FastAPI endpoint and call it from your Dart code
using the `http` package. The model outputs clean JSON — no parsing gymnastics needed.
